# Tutorial 3 — WorldView-3 Mount Soledad, three images blended

> Reading this as a static page? To run the notebook, [open this repo in a GitHub Codespace](https://stereopipeline-quickstart.readthedocs.io/en/latest/start/codespaces.html); ASP and `asp-plot` come pre-installed.

[Tutorial 2](02_worldview_ucsd.ipynb) made a DEM from one pair of images. With a third image of the same place there are three pairs, and three DEMs that each miss different spots and carry different noise. Averaging them gives one DEM with fewer holes and less noise. Studies of this image stack found that blending pairs this way gave a more accurate DEM than any single pair, and than ASP's joint multi-view triangulation of all images at once.

This tutorial uses the same 3 × 3 km window as Tutorial 2, adds a third WorldView-3 image, and runs stereo on all three pairs. It skips orthorectification: on these small crops, stereo on the cropped images directly is as accurate and saves a step. In a 4-core Codespace it runs top to bottom in about 16 minutes after Tutorial 2, or about 20 if Tutorial 2 hasn't downloaded and cropped the first two images.

> To run it, click **Run All** at the top, then scroll down to follow it: each step's cell shows its progress while it runs, then its results. The code cells are collapsed to one line; click one to see its code.

| Catid | Date | Off-nadir |
|---|---|---|
| `104001000496A100` | 2014-11-09 | 1.1° |
| `10400100071D8800` | 2015-01-24 | 16.3° |
| `10400100057DD500` | 2014-12-23 | 24.1° |

The pairs converge at 16.9°, 25.9° and 25.3°.

## What we'll do

1. Download the three images (the first two are reused from Tutorial 2).
2. Plot the stereo geometry.
3. Crop all three to the window.
4. Bundle adjust the three cameras together.
5. Run stereo on each pair and make three DEMs.
6. Blend the three DEMs with `dem_mosaic`.
7. Make a PDF report of the blended DEM, including a comparison with ICESat-2.

While a step runs, its cell shows the command, the last few lines of ASP's output and a status line with the time so far. ASP's full output goes to a log file in `data/ucsd_soledad/logs/`.

In [ ]:
from pathlib import Path

from utils import run, run_stereo, timer

DATA = "../data/ucsd_soledad"
LOGS = f"{DATA}/logs"

!mkdir -p {DATA}

# The processing window, xmin ymin xmax ymax in UTM zone 11N (EPSG:32611)
# meters: 3 x 3 km around the summit of Mount Soledad.
T_SRS = "EPSG:32611"
T_PROJWIN = "476000 3632000 479000 3635000"

# The Copernicus reference DEM, fetched by the download step.
REF_DEM = f"{DATA}/ref/cop30_soledad.tif"

# Crops are averaged FACTOR x FACTOR pixels: 4 turns the native ~0.31 m
# pixels into ~1.2 m ones, which keeps stereo to a few minutes.
FACTOR = 4

SCENES = ["104001000496A100", "10400100071D8800", "10400100057DD500"]
PAIRS = [(SCENES[0], SCENES[1]), (SCENES[0], SCENES[2]), (SCENES[2], SCENES[1])]

FRESH_ICESAT2 = False
SHIPPED_SELECTIONS = f"{DATA}/icesat2_shipped/blend_figure_selections.yml"

## 1. Download the images

The same download script as Tutorial 2, now with all three scenes. Images already downloaded are skipped. The imagery is IARPA CORE3D data, released for public research and downloaded from SpaceNet's public AWS bucket. Publications using it should cite Hagstrom et al. (2021), "Cumulative assessment for urban 3D modeling" (IEEE IGARSS).

> In a 4-core Codespace, this step takes under a minute after Tutorial 2, or about 2 minutes without it.

In [ ]:
run(f"bash ../scripts/download_worldview_ucsd.sh {DATA} {' '.join(SCENES)}", log=f"{LOGS}/download.log", label="Downloading the images")

## 2. Stereo geometry

> In a 4-core Codespace, this step takes about half a minute the first time, most of it loading `asp-plot` and its libraries; the background map tiles were fetched when the Codespace was built. Elsewhere, the first run also fetches the tiles, which can take up to a minute.

In [ ]:
with timer("Plotting the stereo geometry"):
    import contextily
    from asp_plot.stereo_geometry import StereoGeometryPlotter

    # The map tiles were fetched when the Codespace was built
    # (scripts/prefetch_basemap.py); without them, this fetches them itself.
    contextily.set_cache_dir("../data/basemap_cache")

    geometry = StereoGeometryPlotter(inputs=[f"{DATA}/{catid}_P001.xml" for catid in SCENES])
    geometry.stereo_geom_plot()

The three view directions spread around the sky: one near the zenith and two from different sides. Each pair sees the ground from a different angle, so a spot hidden from one pair is often visible to another.

## 3. Crop to the window

The same crop as Tutorial 2: `cam2rpc` cuts out the window and fits an RPC camera, and `average_crop()` averages to 1.2 m pixels. Crops already made by Tutorial 2 are skipped.

> In a 4-core Codespace, this step takes under a minute after Tutorial 2, or about 2 minutes without it.

In [ ]:
from utils import average_crop

Path(f"{DATA}/crops/full").mkdir(parents=True, exist_ok=True)
xmin, ymin, xmax, ymax = map(float, T_PROJWIN.split())
RPC_BOX = f"{DATA}/crops/rpc_box.tif"
if not Path(RPC_BOX).exists():
    # The reference DEM over the window plus 250 m: the ground cam2rpc fits over.
    !gdal_translate -q -projwin {xmin - 250} {ymax + 250} {xmax + 250} {ymin - 250} {REF_DEM} {RPC_BOX}

for catid in SCENES:
    if Path(f"{DATA}/crops/{catid}_P001.tif").exists():
        print(f"{DATA}/crops/{catid}_P001.tif exists — skipping. Delete it to recrop.")
        continue
    run(f"""
        cam2rpc -t dg --dem-file {RPC_BOX} --save-tif-image \
            {DATA}/{catid}_P001.NTF {DATA}/{catid}_P001.xml \
            {DATA}/crops/full/{catid}.xml
        """,
        log=f"{LOGS}/cam2rpc_{catid}.log",
        label=f"Cropping {catid}",
    )
    with timer(f"Averaging {catid} to {FACTOR} x {FACTOR} pixels"):
        average_crop(f"{DATA}/crops/full/{catid}.tif", f"{DATA}/crops/full/{catid}.xml",
                     f"{DATA}/crops/{catid}_P001.tif", f"{DATA}/crops/{catid}_P001.xml", FACTOR)

!ls -lh {DATA}/crops/*.tif

## 4. Bundle adjustment

All three cameras are adjusted together, so every pair is triangulated with cameras that agree, again held to the reference DEM's heights. The result goes under `ba3/`, separate from Tutorial 2's two-camera solution.

> In a 4-core Codespace, this step takes under a minute.

In [ ]:
if Path(f"{DATA}/ba3/run-final_residuals_pointmap.csv").exists():
    print(f"{DATA}/ba3/run-final_residuals_pointmap.csv exists — skipping bundle_adjust. Delete {DATA}/ba3/ to reprocess.")
else:
    images = " ".join(f"{DATA}/crops/{c}_P001.tif" for c in SCENES)
    cameras = " ".join(f"{DATA}/crops/{c}_P001.xml" for c in SCENES)
    run(f"""
        bundle_adjust -t rpc --threads 4 \
            --heights-from-dem {Path(REF_DEM).resolve()} --heights-from-dem-uncertainty 10 \
            {images} {cameras} -o {DATA}/ba3/run
        """,
        log=f"{LOGS}/bundle_adjust_3.log",
        label="Bundle adjusting the three cameras",
    )

!cat {DATA}/ba3/run-initial_residuals_stats.txt {DATA}/ba3/run-final_residuals_stats.txt

The reprojection error per image, before and after: the median drops from about a pixel to about 0.2 pixels for each of the three.

## 5. Stereo on each pair

Stereo runs on the cropped images directly (`-t rpc`), without orthorectifying them first. ASP then aligns each pair itself, with a simple transform that is enough over a 3 km window. The other settings are as in Tutorial 2.

For each pair, the cell shows the matched features and then the map, as in Tutorial 2. The map is the left image as ASP aligned it to the right one, so it may be rotated. The images aren't orthorectified, so the shifts on the left are larger here: tens of pixels, from the different viewing angles.

> In a 4-core Codespace, this step takes about 15 minutes, 3½ to 5½ minutes per pair.

In [ ]:
def pair_dir(left, right):
    return f"{DATA}/pairs/{left}_{right}"

for n, (left, right) in enumerate(PAIRS, start=1):
    out = pair_dir(left, right)
    if Path(f"{out}/run-DEM.tif").exists():
        print(f"{out}/run-DEM.tif exists — skipping. Delete {out}/ to reprocess.")
        continue
    run_stereo(f"""
        parallel_stereo -t rpc \
            --stereo-algorithm asp_mgm --subpixel-mode 9 \
            --job-size-w 4096 --job-size-h 4096 \
            --processes 1 --threads-multiprocess 4 --threads-singleprocess 4 \
            --bundle-adjust-prefix {DATA}/ba3/run \
            {DATA}/crops/{left}_P001.tif {DATA}/crops/{right}_P001.tif \
            {DATA}/crops/{left}_P001.xml {DATA}/crops/{right}_P001.xml \
            {out}/run
        """,
        prefix=f"{out}/run",
        log=f"{LOGS}/parallel_stereo_pair{n}.log",
        label=f"Stereo, pair {n} of 3",
    )
    run(
        f"point2dem --threads 4 --tr 2.4 --t_srs {T_SRS} --errorimage {out}/run-PC.tif",
        log=f"{LOGS}/point2dem_pair{n}.log",
        label=f"Gridding pair {n} into a DEM",
    )

In [ ]:
from utils import show_hillshades

show_hillshades([f"{pair_dir(l, r)}/run-DEM.tif" for l, r in PAIRS],
                ["nadir + 16°", "nadir + 24°", "24° + 16°"])

The three pair DEMs as hillshades. They agree on the landforms, but each has its own holes (white) and its own speckle. The two wider pairs see the steep canyon walls from more of an angle and show more texture there.

## 6. Blend the three DEMs

`dem_mosaic` averages the three DEMs cell by cell. Where a DEM has a hole, the others fill it.

> This step takes a few seconds.

In [ ]:
BLEND = f"{DATA}/pairs/{PAIRS[0][0]}_{PAIRS[0][1]}/run-blend-DEM.tif"
if Path(BLEND).exists():
    print(f"{BLEND} exists — skipping dem_mosaic. Delete it to reblend.")
else:
    dems = " ".join(f"{pair_dir(l, r)}/run-DEM.tif" for l, r in PAIRS)
    run(f"dem_mosaic --threads 4 {dems} -o {BLEND}", log=f"{LOGS}/dem_mosaic.log", label="Blending the three DEMs")

show_hillshades([f"{pair_dir(*PAIRS[0])}/run-DEM.tif", BLEND],
                ["One pair (Tutorial 2's, here without orthorectification)", "Three pairs blended"])

Left, one pair; right, the blend. The blend fills nearly every hole and the speckle averages out, while ridges, roofs and roads stay sharp.

The blend is saved inside the first pair's stereo folder, so the report in the next step can use that pair's stereo results for its stereo pages and the blend for its DEM pages.

## 7. Report

The same `asp_report` as Tutorial 2, pointed at the blended DEM with `--dem-filename`. The report's stereo pages (matches, disparity) come from the first pair; its DEM, hillshade and ICESat-2 pages show the blend.

> In a 4-core Codespace, this step takes under a minute.

In [ ]:
from utils import show_report_figures

STEREO = f"pairs/{PAIRS[0][0]}_{PAIRS[0][1]}"
FIGS = f"{DATA}/{STEREO}/report_figures"
if Path(f"{DATA}/{STEREO}/soledad_wv3_blend_report.pdf").exists():
    print(f"{DATA}/{STEREO}/soledad_wv3_blend_report.pdf exists — skipping asp_report. Delete it to regenerate.")
else:
    reuse = "" if FRESH_ICESAT2 else f"--reuse-selections {SHIPPED_SELECTIONS}"
    run(f"""
        asp_report \
            --directory {DATA} \
            --bundle-adjust-prefix ba3 \
            --stereo-directory {STEREO} \
            --dem-filename run-blend-DEM.tif \
            --reference-dem {REF_DEM} \
            --subset-km 0.5 \
            {reuse} \
            --figures-directory {FIGS} \
            --report-filename soledad_wv3_blend_report.pdf
        """,
        log=f"{LOGS}/asp_report_blend.log",
        label="Making the report",
    )

show_report_figures(FIGS)

The report is in `data/ucsd_soledad/pairs/104001000496A100_10400100071D8800/soledad_wv3_blend_report.pdf`. To save a copy to your computer, right-click the PDF in the file explorer on the left and choose **Download**. Compare it with Tutorial 2's report: the ICESat-2 profile follows the same track.

## 8. Download the results

To look at the results in QGIS, ArcGIS or another GIS, the cell below puts them in one zip file, `soledad_blend_results.zip`: the report, the blended DEM, and each pair's DEM and map of triangulation error, in a folder per pair. When it's ready, a notification at the bottom right points to it in the file explorer; right-click it there and choose **Download**.

> This step takes a few seconds.

In [ ]:
from utils import zip_results

zip_results(
    f"{DATA}/soledad_blend_results.zip",
    [f"{DATA}/{STEREO}/soledad_wv3_blend_report.pdf", BLEND]
    + [f"{pair_dir(l, r)}/run-{name}.tif" for l, r in PAIRS for name in ("DEM", "IntersectionErr")],
)